# CVAE
- Script for analysing mean data extracted from Kinematrix
- Mean model with hold-out-set

In [ ]:
#%% Load packages

import pandas as pd
print(pd.__version__)
import numpy as np
print(np.__version__)
import matplotlib.pyplot as plt
#import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torch.nn.functional as F
import plotly.express as px
from pandas import read_csv
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from sklearn.preprocessing import OneHotEncoder
import umap
from scipy.spatial.distance import cdist
import plotly.express as px
import plotly.io as pio
from collections import defaultdict, Counter
#from upsetplot import from_memberships, UpSet
from sklearn.cluster import DBSCAN
from matplotlib.patches import Patch
from scipy.stats import f_oneway, kruskal, shapiro, levene
#from statsmodels.stats.multitest import multipletests
#import scikit_posthocs as sp
from itertools import combinations

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support,
    f1_score, roc_auc_score, average_precision_score
)
import copy
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler



In [ ]:
#%% Injury severity

#################
### Load data ###
#################
# 1. Load the fully preprocessed dataset (contains Cluster and Asym columns)
df_scaled_with_asym = pd.read_csv("/Users/asm/Desktop/UNI/MasterThesis/data/final/cvae_data/preprocessed_datafile_cvae.csv")

# Remove specific mouse if needed
# df_scaled = df_scaled[df_scaled['Mouse'] != 182].reset_index(drop=True)

#######################
### Prepare Tensors ###
#######################

# 2. Create Asymmetry Tensor directly from the existing columns
# No need to re-run the functions; the columns are already in the CSV.
asymmetry_cols = ["Asym_L", "Asym_R", "Asym_B"]
asymmetry_tensor = torch.tensor(
    df_scaled_with_asym[asymmetry_cols].values.astype(np.float32)
)

# 3. Create Features (x) and Labels (y)
# CRITICAL: We must drop the metadata and the Asym/Cluster columns so they aren't features.
cols_to_drop = [
    'Dataset', 'Mouse', 'Person', 'Injury', 'Injury_Label', # Metadata
    'Cluster', 'Asym_L', 'Asym_R', 'Asym_B'                 # Calculated Covariates
]

x_array = df_scaled_with_asym.drop(columns=cols_to_drop).values.astype(np.float32)
y_array = df_scaled_with_asym["Injury_Label"].values.astype(int)

# 4. Convert to torch tensors
x_tensor = torch.tensor(x_array, dtype=torch.float32)
y_tensor = torch.tensor(y_array, dtype=torch.long)

#############################
### Split Labeled/Unlabeled #
#############################

# Identify labeled and unlabeled data (-1 is unlabeled)
labeled_mask = y_tensor != -1

# Split everything using the mask
x_labeled = x_tensor[labeled_mask]
y_labeled = y_tensor[labeled_mask]
asym_labeled = asymmetry_tensor[labeled_mask]

x_unlabeled = x_tensor[~labeled_mask]
asym_unlabeled = asymmetry_tensor[~labeled_mask]

print(f"Data Loaded Successfully.")
print(f"Features (x): {x_labeled.shape[1]} dimensions")
print(f"Labeled samples: {len(x_labeled)}")
print(f"Unlabeled samples: {len(x_unlabeled)}")

In [ ]:
#%% ==============================
# Split setup: HOLD-OUT TEST + Repeated Stratified K-Fold on Train/Val
# ==============================

SEED = 42
rng = np.random.default_rng(SEED)

# Global labels in df (one row per mouse in your mean-data setting)
y_all_np = df_scaled_with_asym["Injury_Label"].to_numpy()

# Global index arrays
idx_h_all = np.where(y_all_np == 0)[0]      # Healthy labeled
idx_s_all = np.where(y_all_np == 1)[0]      # Severe labeled
idx_u_all = np.where(y_all_np == -1)[0]     # Unlabeled

idx_labeled_all = np.concatenate([idx_h_all, idx_s_all])
y_labeled_all   = y_all_np[idx_labeled_all].astype(int)

# ------------------------------
# 1) HOLD-OUT TEST SET (labeled only, stratified)
# ------------------------------
idx_labeled_trainval, idx_labeled_test, y_trainval, y_test = train_test_split(
    idx_labeled_all,
    y_labeled_all,
    test_size=0.17,
    random_state=SEED,
    stratify=y_labeled_all
)

idx_unlabeled_trainval, idx_unlabeled_test = train_test_split(
    idx_u_all,
    test_size=0.15,
    random_state=SEED
)

print("HOLD-OUT TEST (labeled, stratified):")
print("  Train/Val labeled:", len(idx_labeled_trainval), "| Test labeled:", len(idx_labeled_test))
print("  Unlabeled Train/Val:", len(idx_unlabeled_trainval), "| Unlabeled Test:", len(idx_unlabeled_test))

# Helpful counts
y_test_np = y_all_np[idx_labeled_test]
print("  Test labeled class counts:", dict(zip(*np.unique(y_test_np, return_counts=True))))

# ------------------------------
# 2) Repeated Stratified K-Fold on labeled Train/Val
# ------------------------------
n_splits = 5
n_repeats = 2
rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=SEED)

# We'll store fold indices (in GLOBAL index space)
folds = []
for fold_id, (train_sub_idx, val_sub_idx) in enumerate(rskf.split(idx_labeled_trainval, y_trainval), start=1):
    idx_train_fold = idx_labeled_trainval[train_sub_idx]
    idx_val_fold   = idx_labeled_trainval[val_sub_idx]

    # sanity check: stratification
    y_tr = y_all_np[idx_train_fold]
    y_va = y_all_np[idx_val_fold]

    folds.append((idx_train_fold, idx_val_fold))
    print(f"Fold {fold_id:02d}: train={len(idx_train_fold)} val={len(idx_val_fold)} | "
          f"train counts={dict(zip(*np.unique(y_tr, return_counts=True)))} | "
          f"val counts={dict(zip(*np.unique(y_va, return_counts=True)))}")

# Fixed (shared) unlabeled training pool for all folds:
idx_unlabeled_fold_train = idx_unlabeled_trainval


In [ ]:
#%% Define CVAE model

class CVAE(nn.Module): 
    def __init__(self, input_dim=3251, label_dim=2, asymmetry_dim=3, latent_dim=16): 
        super(CVAE, self).__init__()
        self.encoder_fc1 = nn.Linear(input_dim + label_dim + asymmetry_dim, 1024)
        self.encoder_ln1 = nn.LayerNorm(1024)
        self.encoder_fc1_uncond = nn.Linear(input_dim + asymmetry_dim, 1024)
        self.encoder_ln1_uncond = nn.LayerNorm(1024)
        self.encoder_fc2 = nn.Linear(1024, 256)
        self.encoder_ln2 = nn.LayerNorm(256)
        self.encoder_mu = nn.Linear(256, latent_dim)
        self.encoder_logvar = nn.Linear(256, latent_dim)

        self.decoder_fc1 = nn.Linear(latent_dim + label_dim + asymmetry_dim, 256)
        self.decoder_ln1 = nn.LayerNorm(256)
        self.decoder_fc1_uncond = nn.Linear(latent_dim + asymmetry_dim, 256)
        self.decoder_ln1_uncond = nn.LayerNorm(256)
        self.decoder_fc2 = nn.Linear(256, 1024)
        self.decoder_ln2 = nn.LayerNorm(1024)
        self.decoder_out = nn.Linear(1024, input_dim)

        self.classifier_fc1 = nn.Linear(latent_dim, 64)
        self.classifier_fc2 = nn.Linear(64, 32)
        self.classifier_out = nn.Linear(32, label_dim)

    def encode(self, x, y=None, asym=None):
        inputs = [x]
        if y is not None:
            inputs.append(y)
        if asym is not None:
            inputs.append(asym)
        x_cat = torch.cat(inputs, dim=1)
    
        if y is not None:
            h = F.relu(self.encoder_ln1(self.encoder_fc1(x_cat)))
        else:
            h = F.relu(self.encoder_ln1_uncond(self.encoder_fc1_uncond(x_cat)))
    
        h = F.relu(self.encoder_ln2(self.encoder_fc2(h)))
        return self.encoder_mu(h), self.encoder_logvar(h)

    def decode(self, z, y=None, asym=None):
        inputs = [z]
        if y is not None:
            inputs.append(y)
        if asym is not None:
            inputs.append(asym)
        h = torch.cat(inputs, dim=1)
    
        if y is not None:
            h = F.relu(self.decoder_ln1(self.decoder_fc1(h)))        
        else:
            h = F.relu(self.decoder_ln1_uncond(self.decoder_fc1_uncond(h))) 
    
        h = F.relu(self.decoder_ln2(self.decoder_fc2(h)))
        return self.decoder_out(h)


    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def classify(self, z):
        h = F.relu(self.classifier_fc1(z))
        h = F.relu(self.classifier_fc2(h))
        return self.classifier_out(h)

    def forward(self, x, y=None, asym=None):
        mu, logvar = self.encode(x, y, asym)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z, y, asym)
        y_logits = self.classify(z)
        return x_recon, mu, logvar, y_logits

In [ ]:
#%% ==============================
# Helpers: build fold tensors + evaluation metrics
# ==============================

def onehot(y_long, num_classes=2):
    return F.one_hot(y_long, num_classes=num_classes).float()

@torch.no_grad()
def predict_probs_from_mu(model, x, asym):
    """
    Matches your training choice: classifier uses mu.
    Returns:
      probs: (N, 2) softmax probabilities
      preds: (N,) predicted class
    """
    model.eval()
    mu, logvar = model.encode(x, y=None, asym=asym)  # encode without label for prediction
    logits = model.classify(mu)
    probs = torch.softmax(logits, dim=1)
    preds = torch.argmax(probs, dim=1)
    return probs, preds

def compute_metrics(y_true_np, y_pred_np, y_prob_np=None):
    """
    y_true_np: shape (N,)
    y_pred_np: shape (N,)
    y_prob_np: shape (N,2) optional
    """
    out = {}
    out["accuracy"] = accuracy_score(y_true_np, y_pred_np)
    out["balanced_accuracy"] = balanced_accuracy_score(y_true_np, y_pred_np)

    # F1 variants
    out["f1_macro"] = f1_score(y_true_np, y_pred_np, average="macro")
    out["f1_weighted"] = f1_score(y_true_np, y_pred_np, average="weighted")
    out["f1_severe"] = f1_score(y_true_np, y_pred_np, pos_label=1)

    # precision/recall for Severe (class 1)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true_np, y_pred_np, average=None, labels=[0,1], zero_division=0)
    out["precision_severe"] = float(prec[1])
    out["recall_severe"] = float(rec[1])

    out["confusion_matrix"] = confusion_matrix(y_true_np, y_pred_np)

    # Prob-based (if available)
    if y_prob_np is not None and y_prob_np.shape[1] == 2:
        # use probability of Severe class
        p_severe = y_prob_np[:, 1]
        try:
            out["roc_auc"] = roc_auc_score(y_true_np, p_severe)
        except ValueError:
            out["roc_auc"] = np.nan
        try:
            out["pr_auc"] = average_precision_score(y_true_np, p_severe)
        except ValueError:
            out["pr_auc"] = np.nan

    return out


In [ ]:
def downsample_to_balance(idx_set, y_all_np, seed=42):
    rng = np.random.default_rng(seed)
    y = y_all_np[idx_set]

    idx_h = idx_set[y == 0]
    idx_s = idx_set[y == 1]

    if len(idx_h) == 0 or len(idx_s) == 0:
        return idx_set  # cannot balance

    n = min(len(idx_h), len(idx_s))
    idx_h_sel = rng.choice(idx_h, size=n, replace=False)

    idx_bal = np.concatenate([idx_h_sel, idx_s])
    rng.shuffle(idx_bal)
    return idx_bal


In [ ]:
# Train one fold with downsampled samples

def train_one_fold_downsampled(
    fold_idx,
    model,
    optimizer,
    idx_train_fold,
    idx_val_fold,
    idx_unlabeled_train,
    x_tensor, y_tensor, asymmetry_tensor,
    y_all_np,
    epochs=100000,
    patience=35,
    alpha=0.5,
    beta=1.0,
    free_bits=0.5,
    batch_size_labeled=64,
    batch_size_unlabeled=64,
    device="cpu",
    seed=42,
    lr_plateau_factor=0.5,
    lr_plateau_patience=5,
    min_lr=1e-7
):
    # ------------------------------
    # Learning-rate scheduler (per fold)
    # ------------------------------
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=lr_plateau_factor,
        patience=lr_plateau_patience,
        min_lr=min_lr,
        verbose=True
    )

    # ------------------------------
    # 1) Downsample TRAIN labeled indices (Healthy -> match Severe)
    # ------------------------------
    idx_train_bal = downsample_to_balance(idx_train_fold, y_all_np, seed=seed)

    # Build labeled train tensors (BALANCED)
    x_l_tr = x_tensor[idx_train_bal].to(device)
    y_l_tr = y_tensor[idx_train_bal].to(device)
    a_l_tr = asymmetry_tensor[idx_train_bal].to(device)

    # Build labeled val tensors (UNCHANGED)
    x_l_va = x_tensor[idx_val_fold].to(device)
    y_l_va = y_tensor[idx_val_fold].to(device)
    a_l_va = asymmetry_tensor[idx_val_fold].to(device)
    y_l_va_oh = onehot(y_l_va, 2)

    # Unlabeled train tensors
    x_u_tr = x_tensor[idx_unlabeled_train].to(device)
    a_u_tr = asymmetry_tensor[idx_unlabeled_train].to(device)

    # Labeled loader
    train_dataset_l = TensorDataset(x_l_tr, y_l_tr, a_l_tr)
    bs_l = min(batch_size_labeled, len(y_l_tr))
    labeled_loader = DataLoader(train_dataset_l, batch_size=bs_l, shuffle=True, drop_last=False)

    best_val = float("inf")
    best_state = None
    counter = 0

    history = {
        "train_total": [], "train_recon": [], "train_kl": [], "train_clf": [],
        "val_total": [], "val_recon": [], "val_kl": [], "val_clf": [],
        "val_acc": [], "val_bal_acc": [], "val_f1_macro": [], "val_f1_severe": [],
        "lr": []
    }

    for epoch in range(epochs):
        model.train()

        epoch_train_total = 0.0
        epoch_train_recon = 0.0
        epoch_train_kl    = 0.0
        epoch_train_clf   = 0.0
        n_batches = 0

        for xb_l, yb_l, ab_l in labeled_loader:
            xb_l = xb_l.to(device)
            yb_l = yb_l.to(device)
            ab_l = ab_l.to(device)

            optimizer.zero_grad()
            yb_l_oh = onehot(yb_l, 2)

            mu_l, logvar_l = model.encode(xb_l, yb_l_oh, asym=ab_l)
            z_l = model.reparameterize(mu_l, logvar_l)
            x_recon_l = model.decode(z_l, yb_l_oh, asym=ab_l)

            logits_l = model.classify(mu_l)
            clf_loss = F.cross_entropy(logits_l, yb_l)

            # Unlabeled batch
            bs = xb_l.size(0)
            ru = torch.randint(0, x_u_tr.size(0), (min(bs, batch_size_unlabeled),), device=device)
            xb_u = x_u_tr[ru]
            ab_u = a_u_tr[ru]

            mu_u, logvar_u = model.encode(xb_u, y=None, asym=ab_u)
            z_u = model.reparameterize(mu_u, logvar_u)
            x_recon_u = model.decode(z_u, y=None, asym=ab_u)

            x_recon = torch.cat([x_recon_l, x_recon_u], dim=0)
            x_target = torch.cat([xb_l, xb_u], dim=0)
            recon_loss = ((x_recon - x_target) ** 2).mean()

            mu = torch.cat([mu_l, mu_u], dim=0)
            logvar = torch.cat([logvar_l, logvar_u], dim=0)
            kl_per_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
            kl_per_dim = torch.clamp(kl_per_dim, min=free_bits)
            kl_loss = kl_per_dim.mean()

            loss = recon_loss + beta * kl_loss + alpha * clf_loss
            loss.backward()
            optimizer.step()

            epoch_train_total += loss.item()
            epoch_train_recon += recon_loss.item()
            epoch_train_kl    += kl_loss.item()
            epoch_train_clf   += clf_loss.item()
            n_batches += 1

        train_total = epoch_train_total / max(n_batches, 1)
        train_recon = epoch_train_recon / max(n_batches, 1)
        train_kl    = epoch_train_kl    / max(n_batches, 1)
        train_clf   = epoch_train_clf   / max(n_batches, 1)

        history["train_total"].append(train_total)
        history["train_recon"].append(train_recon)
        history["train_kl"].append(train_kl)
        history["train_clf"].append(train_clf)

        # ---- Validation ----
        model.eval()
        with torch.no_grad():
            mu_va, logvar_va = model.encode(x_l_va, y_l_va_oh, asym=a_l_va)
            z_va = model.reparameterize(mu_va, logvar_va)
            x_recon_va = model.decode(z_va, y_l_va_oh, asym=a_l_va)

            logits_va = model.classify(mu_va)
            clf_va = F.cross_entropy(logits_va, y_l_va)

            recon_va = ((x_recon_va - x_l_va) ** 2).mean()

            kl_va_per_dim = -0.5 * (1 + logvar_va - mu_va.pow(2) - logvar_va.exp())
            kl_va_per_dim = torch.clamp(kl_va_per_dim, min=free_bits)
            kl_va = kl_va_per_dim.mean()

            val_loss = recon_va + beta * kl_va + alpha * clf_va

            probs_va = torch.softmax(logits_va, dim=1).cpu().numpy()
            preds_va = np.argmax(probs_va, axis=1)
            y_va_np = y_l_va.cpu().numpy()
            val_metrics = compute_metrics(y_va_np, preds_va, probs_va)

        # Step scheduler AFTER val loss is computed
        scheduler.step(val_loss)

        # Track LR
        current_lr = optimizer.param_groups[0]["lr"]
        history["lr"].append(float(current_lr))

        history["val_total"].append(float(val_loss.item()))
        history["val_recon"].append(float(recon_va.item()))
        history["val_kl"].append(float(kl_va.item()))
        history["val_clf"].append(float(clf_va.item()))
        history["val_acc"].append(val_metrics["accuracy"])
        history["val_bal_acc"].append(val_metrics["balanced_accuracy"])
        history["val_f1_macro"].append(val_metrics["f1_macro"])
        history["val_f1_severe"].append(val_metrics["f1_severe"])

        # Early stopping
        if val_loss.item() < best_val:
            best_val = val_loss.item()
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

        print(
            f"Fold {fold_idx} | Epoch {epoch+1}/{epochs} | "
            f"LR: {current_lr:.6f} | "
            f"Train Loss: {train_total:.4f} | Recon: {train_recon:.4f} | KL: {train_kl:.4f} | Clf: {train_clf:.4f} | "
            f"Val Loss: {val_loss.item():.4f} | Val F1(severe): {val_metrics['f1_severe']:.3f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, best_val, idx_train_bal


In [ ]:
def count_labels(indices, y_all_np):
    vals = y_all_np[indices]
    uniq, cnt = np.unique(vals, return_counts=True)
    return dict(zip(uniq.tolist(), cnt.tolist()))

In [ ]:
#%% ==============================
# Run repeated stratified K-fold CV + Final test evaluation (TRAIN downsampling only)
# ==============================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

SEED = 42
alpha = 0.5
beta = 1.0
free_bits = 0.5
epochs = 100000
patience = 35

# ---- Cross-validation ----
cv_results = []
cv_best_vals = []
fold_histories = []
fold_train_balanced_counts = []  # optional: store downsampled counts per fold

for fold_id, (idx_train_fold, idx_val_fold) in enumerate(folds, start=1):
    model = CVAE(input_dim=3251).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


    # Downsample TRAIN only (inside the function), keep VAL untouched
    model, hist, best_val, idx_train_bal = train_one_fold_downsampled(
        fold_idx=fold_id,
        model=model,
        optimizer=optimizer,
        idx_train_fold=idx_train_fold,
        idx_val_fold=idx_val_fold,
        idx_unlabeled_train=idx_unlabeled_fold_train,
        x_tensor=x_tensor,
        y_tensor=y_tensor,
        asymmetry_tensor=asymmetry_tensor,
        y_all_np=y_all_np,
        epochs=epochs,
        patience=patience,
        alpha=alpha,
        beta=beta,
        free_bits=free_bits,
        device=device,
        seed=SEED
    )

    fold_histories.append(hist)

    # Optional: store counts for reporting
    fold_train_balanced_counts.append(count_labels(idx_train_bal, y_all_np))

    # Evaluate fold on ORIGINAL validation set (unbalanced / natural)
    with torch.no_grad():
        x_va = x_tensor[idx_val_fold].to(device)
        y_va = y_tensor[idx_val_fold].to(device)
        a_va = asymmetry_tensor[idx_val_fold].to(device)

        probs_va, preds_va = predict_probs_from_mu(model, x_va, a_va)

        y_va_np_local = y_va.detach().cpu().numpy()
        preds_va_np = preds_va.detach().cpu().numpy()
        probs_va_np = probs_va.detach().cpu().numpy()

        metrics_va = compute_metrics(y_va_np_local, preds_va_np, probs_va_np)

    cv_results.append(metrics_va)
    cv_best_vals.append(best_val)

    print(
        f"[Fold {fold_id:02d}] best val loss={best_val:.4f} | "
        f"F1_macro={metrics_va['f1_macro']:.3f} | "
        f"F1_severe={metrics_va['f1_severe']:.3f} | "
        f"BalAcc={metrics_va['balanced_accuracy']:.3f} | "
        f"Train_balanced_counts={count_labels(idx_train_bal, y_all_np)}"
    )


# ---- Summarize CV ----
def summarize_cv(key):
    vals = [m[key] for m in cv_results if key in m and m[key] is not None]
    return float(np.mean(vals)), float(np.std(vals))

print("\n=== Repeated Stratified K-Fold CV Summary ===")
for k in ["balanced_accuracy", "f1_macro", "f1_severe", "precision_severe", "recall_severe", "roc_auc", "pr_auc"]:
    mean_k, std_k = summarize_cv(k)
    print(f"{k:>18s}: {mean_k:.3f} ± {std_k:.3f}")


# ---- Train FINAL model on all labeled Train/Val + all unlabeled Train/Val ----
idx_final_train, idx_final_val, y_final_train, y_final_val = train_test_split(
    idx_labeled_trainval, y_trainval,
    test_size=0.15,
    random_state=SEED,
    stratify=y_trainval
)

final_model = CVAE(input_dim=3251).to(device)
final_optimizer = torch.optim.Adam(final_model.parameters(), lr=1e-3)

# Downsample TRAIN only, keep VAL untouched
final_model, final_hist, final_best_val, idx_final_train_bal = train_one_fold_downsampled(
    fold_idx="FINAL",
    model=final_model,
    optimizer=final_optimizer,
    idx_train_fold=idx_final_train,
    idx_val_fold=idx_final_val,
    idx_unlabeled_train=idx_unlabeled_trainval,
    x_tensor=x_tensor,
    y_tensor=y_tensor,
    asymmetry_tensor=asymmetry_tensor,
    y_all_np=y_all_np,
    epochs=epochs,
    patience=patience,
    alpha=alpha,
    beta=beta,
    free_bits=free_bits,
    device=device,
    seed=SEED
)

# Save final model
save_path = "/Users/asm/Desktop/UNI/MasterThesis/holdout_cvae_final_model.pth"
torch.save(final_model.state_dict(), save_path)
print("\nSaved final model to:", save_path)


# ---- Evaluate on HELD-OUT TEST (labeled) ----
x_te = x_tensor[idx_labeled_test].to(device)
y_te = y_tensor[idx_labeled_test].to(device)
a_te = asymmetry_tensor[idx_labeled_test].to(device)

with torch.no_grad():
    probs_te, preds_te = predict_probs_from_mu(final_model, x_te, a_te)
    y_te_np = y_te.detach().cpu().numpy()
    preds_te_np = preds_te.detach().cpu().numpy()
    probs_te_np = probs_te.detach().cpu().numpy()

test_metrics = compute_metrics(y_te_np, preds_te_np, probs_te_np)

print("\n=== HELD-OUT TEST RESULTS (Labeled only) ===")
print("Accuracy          :", f"{test_metrics['accuracy']:.3f}")
print("Balanced Accuracy :", f"{test_metrics['balanced_accuracy']:.3f}")
print("F1 macro          :", f"{test_metrics['f1_macro']:.3f}")
print("F1 severe         :", f"{test_metrics['f1_severe']:.3f}")
print("Precision severe  :", f"{test_metrics['precision_severe']:.3f}")
print("Recall severe     :", f"{test_metrics['recall_severe']:.3f}")
print("ROC AUC           :", f"{test_metrics.get('roc_auc', np.nan):.3f}")
print("PR AUC            :", f"{test_metrics.get('pr_auc', np.nan):.3f}")
print("Confusion matrix:\n", test_metrics["confusion_matrix"])
print("\nClassification report:\n", classification_report(y_te_np, preds_te_np, digits=3))

# Final split reporting
print("\nFINAL TRAIN (original) counts:", count_labels(idx_final_train, y_all_np))
print("FINAL TRAIN (downsampled) counts:", count_labels(idx_final_train_bal, y_all_np))
print("FINAL VAL (natural) counts:", count_labels(idx_final_val, y_all_np))
print("FINAL TEST (natural) counts:", count_labels(idx_labeled_test, y_all_np))


In [ ]:
#%% ==============================
# Plot Train vs Val curves (averaged across folds)
# ==============================


import numpy as np
import matplotlib.pyplot as plt

def pad_to_max(list_of_lists, pad_value=np.nan):
    max_len = max(len(x) for x in list_of_lists)
    arr = np.full((len(list_of_lists), max_len), pad_value, dtype=float)
    for i, seq in enumerate(list_of_lists):
        arr[i, :len(seq)] = seq
    return arr


def plot_train_val(train_key, val_key, title, ylabel):
    train_series = [h[train_key] for h in fold_histories if train_key in h]
    val_series   = [h[val_key]   for h in fold_histories if val_key in h]

    train_arr = pad_to_max(train_series)
    val_arr   = pad_to_max(val_series)

    train_mean = np.nanmean(train_arr, axis=0)
    train_std  = np.nanstd(train_arr, axis=0)

    val_mean = np.nanmean(val_arr, axis=0)
    val_std  = np.nanstd(val_arr, axis=0)

    x = np.arange(len(train_mean))

    plt.figure(figsize=(12,6))

    # Train line
    plt.plot(x, train_mean, linewidth=2, label="Train Loss")
    plt.fill_between(x, train_mean, train_mean, alpha=0.15)

    # Validation line
    plt.plot(x, val_mean, color="orange", linestyle="--", linewidth=2, label="Validation Loss")
    plt.fill_between(x, val_mean, val_mean,
                     color="orange", alpha=0.15)

    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


plot_train_val("train_total", "val_total",
               "Averaged Total loss over Epochs", "Loss")

plot_train_val("train_recon", "val_recon",
               "Reconstruction Loss", "Loss")

plot_train_val("train_kl", "val_kl",
               "KL Divergence Loss", "Loss")

plot_train_val("train_clf", "val_clf",
               "Classification Loss", "Loss")

# plot_train_val("train_total", "val_total",
#                "Total loss", "Loss")

# plot_train_val("train_total", "val_total",
#                "Total loss", "Loss")

# plot_train_val("train_total", "val_total",
#                "Total loss", "Loss")

# plot_train_val("train_total", "val_total",
#                "Total loss", "Loss")

# F1 only exists for validation (no train F1)
plot_train_val("val_f1_macro", "val_f1_macro",
               "Validation F1 (Accuracy)", "F1")

plot_train_val("val_f1_severe", "val_f1_severe",
               "Validation F1 Severe", "F1")




In [ ]:
# ==========================================
# EXTRACT EXACT NUMBERS FOR THESIS TEXT
# ==========================================
import numpy as np

# 1. Get the list of total losses from history
train_total_list = [h['train_total'] for h in fold_histories]
val_total_list   = [h['val_total'] for h in fold_histories]

# 2. Pad them so they are all the same length (to calculate average)
def get_max_len(lst):
    return max(len(x) for x in lst)

max_epochs = get_max_len(train_total_list)

# Create arrays filled with NaN
arr_train = np.full((len(train_total_list), max_epochs), np.nan)
arr_val   = np.full((len(val_total_list),   max_epochs), np.nan)

for i, seq in enumerate(train_total_list):
    arr_train[i, :len(seq)] = seq
    
for i, seq in enumerate(val_total_list):
    arr_val[i, :len(seq)] = seq

# 3. Calculate Averages
mean_train = np.nanmean(arr_train, axis=0)
mean_val   = np.nanmean(arr_val, axis=0)

# 4. Print the "Last Epoch" values
print(f"--- NUMBERS FOR TEXT ---")
print(f"Total Epochs: {max_epochs}")
print(f"Final Avg Train Loss: {mean_train[-1]:.4f}")
print(f"Final Avg Val Loss:   {mean_val[-1]:.4f}")

# Calculate the gap/difference
diff = mean_val[-1] - mean_train[-1]
pct_diff = (diff / mean_train[-1]) * 100
print(f"Difference: {diff:.4f} ({pct_diff:.2f}%)")

import numpy as np

# Helper function to pad lists (handling different lengths)
def get_max_len(lst):
    if not lst: return 0
    return max(len(x) for x in lst)

def pad_and_mean(loss_key, histories):
    # Extract the specific loss list from all folds
    loss_list = [h[loss_key] for h in histories if loss_key in h]
    
    if not loss_list:
        return None, 0

    max_epochs = get_max_len(loss_list)
    arr = np.full((len(loss_list), max_epochs), np.nan)

    for i, seq in enumerate(loss_list):
        arr[i, :len(seq)] = seq
    
    # Calculate mean ignoring NaNs
    return np.nanmean(arr, axis=0), max_epochs

# ==========================================
# EXTRACT COMPONENT LOSSES
# ==========================================

print(f"\n--- FINAL COMPONENT LOSSES (Averaged across folds) ---")

# 1. Reconstruction Loss
mean_train_recon, _ = pad_and_mean('train_recon', fold_histories)
mean_val_recon, _   = pad_and_mean('val_recon', fold_histories)
print(f"Reconstruction -> Train: {mean_train_recon[-1]:.4f} | Val: {mean_val_recon[-1]:.4f}")

# 2. KL Divergence
mean_train_kl, _ = pad_and_mean('train_kl', fold_histories)
mean_val_kl, _   = pad_and_mean('val_kl', fold_histories)
print(f"KL Divergence  -> Train: {mean_train_kl[-1]:.4f} | Val: {mean_val_kl[-1]:.4f}")

# 3. Classification Loss
mean_train_clf, _ = pad_and_mean('train_clf', fold_histories)
mean_val_clf, _   = pad_and_mean('val_clf', fold_histories)
print(f"Classification -> Train: {mean_train_clf[-1]:.4f} | Val: {mean_val_clf[-1]:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Extract data from the final history
t_loss = final_hist['train_total']
v_loss = final_hist['val_total']
epochs = range(1, len(t_loss) + 1)

# 2. Find the "Best" epoch (where validation loss was lowest)
best_epoch = np.argmin(v_loss) + 1
best_val_loss = v_loss[best_epoch-1]
best_train_loss = t_loss[best_epoch-1]

# 3. Plot
plt.figure(figsize=(12, 6))
plt.plot(epochs, t_loss, label='Total Train Loss', linewidth=2)
plt.plot(epochs, v_loss, label='Total Validation Loss', linestyle='--', linewidth=2)

# Mark the best epoch
plt.axvline(best_epoch, color='r', linestyle=':', alpha=0.5, label=f'Best Model (Epoch {best_epoch})')

plt.xlabel('Epochs')
plt.ylabel('Total Loss')
plt.title('Total Loss over Epochs')
plt.legend()
plt.grid(True)
plt.tight_layout()

# Save it
plt.savefig('cvae_learningcurve_final.png')
plt.show()

# 4. Print Exact Numbers for your Text
print("=== NUMBERS FOR THESIS TEXT ===")
print(f"Total Epochs Run: {len(t_loss)}")
print(f"Best Model Saved at Epoch: {best_epoch}")
print(f"Train Loss at Best Epoch: {best_train_loss:.4f}")
print(f"Val Loss at Best Epoch:   {best_val_loss:.4f}")
print(f"Difference: {best_val_loss - best_train_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. SETUP DATA
# ==========================================
# We assume 'final_hist' is available from your previous run
epochs = range(1, len(final_hist['train_total']) + 1)

# Find Best Epoch based on TOTAL Validation Loss (Standard criterion)
# This is the epoch where the model was saved.
best_epoch_idx = np.argmin(final_hist['val_total'])
best_epoch = best_epoch_idx + 1

print(f"=== FINAL MODEL STATS (at Best Epoch {best_epoch}) ===")

# ==========================================
# 2. DEFINE PLOTTING FUNCTION
# ==========================================
def plot_metric(ax, train_key, val_key, title, ylabel, is_loss=True):
    # Extract data
    if train_key in final_hist:
        ax.plot(epochs, final_hist[train_key], label='Train', linewidth=2)
    
    if val_key in final_hist:
        ax.plot(epochs, final_hist[val_key], label='Validation', linestyle='--', linewidth=2)
        
        # Get value at best epoch
        val_at_best = final_hist[val_key][best_epoch_idx]
        if train_key in final_hist:
            train_at_best = final_hist[train_key][best_epoch_idx]
            print(f"{title} -> Train: {train_at_best:.4f} | Val: {val_at_best:.4f}")
        else:
            print(f"{title} -> Val: {val_at_best:.4f}")

    # Vertical line for Best Epoch
    #ax.axvline(best_epoch, color='r', linestyle=':', alpha=0.5, label=f'Best Epoch ({best_epoch})')
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Epochs')
    ax.legend()
    ax.grid(True, alpha=0.3)

# ==========================================
# 3. CREATE PLOTS (2x2 GRID)
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Reconstruction Loss
plot_metric(axes[0, 0], 'train_recon', 'val_recon', 
            "Reconstruction Loss", "MSE")

# Plot 2: KL Divergence
plot_metric(axes[0, 1], 'train_kl', 'val_kl', 
            "KL Divergence", "KLD")

# Plot 3: Classification Loss
plot_metric(axes[1, 0], 'train_clf', 'val_clf', 
            "Classification Loss", "Cross Entropy")

# Plot 4: Accuracy (Balanced)
acc_key = 'val_balanced_accuracy'
# Fallback if the key doesn't exist in your history
if acc_key not in final_hist:
    acc_key = 'val_f1_macro' 

# We manually plot this one to ensure it is ORANGE
if acc_key in final_hist:
    # Plot the line in orange
    axes[1, 1].plot(epochs, final_hist[acc_key], 
                    label='Validation', color='darkorange', linestyle='--', linewidth=2)
    
    # Mark the best epoch vertical line
    #axes[1, 1].axvline(best_epoch, color='r', linestyle=':', alpha=0.5)
    
    # Titles and Labels
    axes[1, 1].set_title("Validation Balanced Accuracy", fontsize=12, fontweight='bold')
    axes[1, 1].set_ylabel("Score (0-1)")
    axes[1, 1].set_xlabel("Epochs")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # Print the value for your thesis text
    val_acc = final_hist[acc_key][best_epoch_idx]
    print(f"Validation Accuracy at Best Epoch: {val_acc:.4f}")

plt.tight_layout()
#plt.savefig('cvae_components_final.png', dpi=300)
plt.show()

In [ ]:
#%% ==============================
# FINAL DATASET NUMBERS FOR REPORT
# ==============================

def count_labels(indices, y_all_np):
    vals = y_all_np[indices]
    uniq, cnt = np.unique(vals, return_counts=True)
    return dict(zip(uniq.tolist(), cnt.tolist()))

y_all_np = df_scaled_with_asym["Injury_Label"].to_numpy()

# Labeled splits
train_counts = count_labels(idx_final_train, y_all_np)
val_counts   = count_labels(idx_final_val, y_all_np)
test_counts  = count_labels(idx_labeled_test, y_all_np)

# Unlabeled
unlab_train_counts = count_labels(idx_unlabeled_trainval, y_all_np)
unlab_test_counts  = count_labels(idx_unlabeled_test, y_all_np)

print("\n--- TRAIN SET ---")
print(f"Total mice: {len(idx_final_train)}")
print(f"Healthy: {train_counts.get(0,0)}")
print(f"Severe : {train_counts.get(1,0)}")

print("\n--- VALIDATION SET ---")
print(f"Total mice: {len(idx_final_val)}")
print(f"Healthy: {val_counts.get(0,0)}")
print(f"Severe : {val_counts.get(1,0)}")

print("\n--- TEST SET (held-out) ---")
print(f"Total mice: {len(idx_labeled_test)}")
print(f"Healthy: {test_counts.get(0,0)}")
print(f"Severe : {test_counts.get(1,0)}")

print("\n--- UNLABELED ---")
print(f"Train unlabeled: {len(idx_unlabeled_trainval)}")
print(f"Test unlabeled : {len(idx_unlabeled_test)}")


In [ ]:
#%% ==============================
# Confusion Matrix Plot (Final CVAE Test Set)
# ==============================

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

# Compute confusion matrix
cm = confusion_matrix(y_te_np, preds_te_np)

# Class labels
class_names = ["Healthy", "Severe"]

plt.figure(figsize=(6,5))

# Plot matrix
plt.imshow(cm, interpolation='nearest', cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()

# Axis ticks
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names)
plt.yticks(tick_marks, class_names)

# Annotate numbers inside cells
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j, i, format(cm[i, j], 'd'),
            horizontalalignment="center",
            color="white" if cm[i, j] > thresh else "black",
            fontsize=12,
        )

plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(6,5))
plt.imshow(cm_norm, interpolation='nearest', cmap="Blues")
plt.title("Normalized Confusion Matrix")
plt.colorbar()

plt.xticks(tick_marks, class_names)
plt.yticks(tick_marks, class_names)

for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        plt.text(
            j, i, f"{cm_norm[i, j]:.2f}",
            horizontalalignment="center",
            color="white" if cm_norm[i, j] > 0.5 else "black",
            fontsize=12,
        )

plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()


In [ ]:
#%% ==============================
# FINAL NUMERIC TRAINING RESULTS
# ==============================

best_epoch = np.argmin(final_hist["val_total"])

final_train_loss = final_hist["train_total"][best_epoch]
final_val_loss   = final_hist["val_total"][best_epoch]

final_recon_train = final_hist["train_recon"][best_epoch]
final_kl_train    = final_hist["train_kl"][best_epoch]
final_clf_train   = final_hist["train_clf"][best_epoch]

final_recon_val = final_hist["val_recon"][best_epoch]
final_kl_val    = final_hist["val_kl"][best_epoch]
final_clf_val   = final_hist["val_clf"][best_epoch]

# Percentage difference train vs validation
loss_diff_percent = ((final_val_loss - final_train_loss) / final_train_loss) * 100

print("\n====================================")
print("FINAL TRAINING NUMBERS (BEST EPOCH)")
print("====================================")

print(
f"Epoch {best_epoch} | "
f"Train Loss: {final_train_loss:.4f} | "
f"Recon: {final_recon_train:.4f} | "
f"KL: {final_kl_train:.4f} | "
f"Clf: {final_clf_train:.4f} | "
f"Val Loss: {final_val_loss:.4f}"
)

print("\nValidation components:")
print(f"Recon: {final_recon_val:.4f}")
print(f"KL   : {final_kl_val:.4f}")
print(f"Clf  : {final_clf_val:.4f}")

print(f"\nTrain vs Val loss difference: {loss_diff_percent:.2f}%")


## Plot and analysis

In [ ]:
import os
import numpy as np
import pandas as pd
import umap
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.express as px
import plotly.io as pio
from sklearn.cluster import DBSCAN
from sklearn.neighbors import KNeighborsClassifier

# ------------------------------
# Settings
# ------------------------------
SEED = 42
rng = np.random.default_rng(SEED)

umap_params_input = dict(
    n_components=3,
    n_neighbors=90,
    min_dist=0.4,
    metric="euclidean",
    random_state=SEED
)

umap_params_supervised = dict(
    n_components=3,
    n_neighbors=90,
    min_dist=0.4,
    metric="euclidean",
    random_state=SEED
)

# ------------------------------
# COLORS
# ------------------------------
C_HEALTHY_TRAIN = "#6495ED"  # light
C_SEVERE_TRAIN  = "#F08080"  # light
C_UNLAB_TRAIN   = "#D3D3D3"  # light gray

C_HEALTHY_PRED  = "#00008B"  # dark
C_SEVERE_PRED   = "#8B0000"  # dark

BG_COLOR        = "#F0F8FF"

# ============================================================
# 0) DEFINE CVAE MODEL (your exact definition)
# ============================================================

class CVAE(nn.Module):
    def __init__(self, input_dim=3251, label_dim=2, asymmetry_dim=3, latent_dim=16):
        super(CVAE, self).__init__()
        self.encoder_fc1 = nn.Linear(input_dim + label_dim + asymmetry_dim, 1024)
        self.encoder_ln1 = nn.LayerNorm(1024)
        self.encoder_fc1_uncond = nn.Linear(input_dim + asymmetry_dim, 1024)
        self.encoder_ln1_uncond = nn.LayerNorm(1024)
        self.encoder_fc2 = nn.Linear(1024, 256)
        self.encoder_ln2 = nn.LayerNorm(256)
        self.encoder_mu = nn.Linear(256, latent_dim)
        self.encoder_logvar = nn.Linear(256, latent_dim)

        self.decoder_fc1 = nn.Linear(latent_dim + label_dim + asymmetry_dim, 256)
        self.decoder_ln1 = nn.LayerNorm(256)
        self.decoder_fc1_uncond = nn.Linear(latent_dim + asymmetry_dim, 256)
        self.decoder_ln1_uncond = nn.LayerNorm(256)
        self.decoder_fc2 = nn.Linear(256, 1024)
        self.decoder_ln2 = nn.LayerNorm(1024)
        self.decoder_out = nn.Linear(1024, input_dim)

        self.classifier_fc1 = nn.Linear(latent_dim, 64)
        self.classifier_fc2 = nn.Linear(64, 32)
        self.classifier_out = nn.Linear(32, label_dim)

    def encode(self, x, y=None, asym=None):
        inputs = [x]
        if y is not None:
            inputs.append(y)
        if asym is not None:
            inputs.append(asym)
        x_cat = torch.cat(inputs, dim=1)

        if y is not None:
            h = F.relu(self.encoder_ln1(self.encoder_fc1(x_cat)))
        else:
            h = F.relu(self.encoder_ln1_uncond(self.encoder_fc1_uncond(x_cat)))

        h = F.relu(self.encoder_ln2(self.encoder_fc2(h)))
        return self.encoder_mu(h), self.encoder_logvar(h)

    def decode(self, z, y=None, asym=None):
        inputs = [z]
        if y is not None:
            inputs.append(y)
        if asym is not None:
            inputs.append(asym)
        h = torch.cat(inputs, dim=1)

        if y is not None:
            h = F.relu(self.decoder_ln1(self.decoder_fc1(h)))
        else:
            h = F.relu(self.decoder_ln1_uncond(self.decoder_fc1_uncond(h)))

        h = F.relu(self.decoder_ln2(self.decoder_fc2(h)))
        return self.decoder_out(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def classify(self, z):
        h = F.relu(self.classifier_fc1(z))
        h = F.relu(self.classifier_fc2(h))
        return self.classifier_out(h)

    def forward(self, x, y=None, asym=None):
        mu, logvar = self.encode(x, y, asym)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z, y, asym)
        y_logits = self.classify(z)
        return x_recon, mu, logvar, y_logits


# ============================================================
# 1) LOAD MODEL (.pth is OrderedDict state_dict)
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
load_path = "/Users/asm/Desktop/UNI/MasterThesis/holdout_cvae_final_model.pth"

print("File exists?", os.path.exists(load_path))
assert os.path.exists(load_path), "ERROR: load_path not found."

model = CVAE(input_dim=3251, label_dim=2, asymmetry_dim=3, latent_dim=16).to(device)
state = torch.load(load_path, map_location=device)  # OrderedDict
model.load_state_dict(state, strict=True)
model.eval()
print("Model loaded on:", next(model.parameters()).device)

# ============================================================
# 2) VERIFY REQUIRED DATA VARIABLES EXIST
# ============================================================

required_vars = [
    "x_tensor", "asymmetry_tensor", "y_all_np", "df_scaled_with_asym",
    "idx_labeled_trainval", "idx_unlabeled_trainval", "idx_labeled_test"
]
missing = [v for v in required_vars if v not in globals()]
assert not missing, f"ERROR: Missing required variables: {missing}\nRun your data prep/split cell first."

# Make sure these are numpy arrays
idx_labeled_trainval = np.array(idx_labeled_trainval)
idx_unlabeled_trainval = np.array(idx_unlabeled_trainval)
idx_test_lbl = np.array(idx_labeled_test)

# ============================================================
# 3) FILTER: Keep ONLY 31 Train/Val Healthy LABELED samples
# ============================================================

trainval_healthy = idx_labeled_trainval[y_all_np[idx_labeled_trainval] == 0]
trainval_severe  = idx_labeled_trainval[y_all_np[idx_labeled_trainval] == 1]

if len(trainval_healthy) < 31:
    raise ValueError(f"Train/Val healthy has only {len(trainval_healthy)} samples, cannot downsample to 31.")

keep_healthy_31 = rng.choice(trainval_healthy, size=31, replace=False)

idx_labeled_trainval_filtered = np.concatenate([keep_healthy_31, trainval_severe])
idx_fit = np.concatenate([idx_labeled_trainval_filtered, idx_unlabeled_trainval])

print("Train/Val Healthy (plotted):", len(keep_healthy_31))
print("Train/Val Severe  (plotted):", len(trainval_severe))
print("Train/Val Unlab   (plotted):", len(idx_unlabeled_trainval))
print("Holdout Test      (plotted):", len(idx_test_lbl))

# ============================================================
# Labels / styles
# ============================================================

def get_label_plot1(row):
    y_val = int(row["y"])
    if y_val == 0: return "Healthy"
    elif y_val == 1: return "Severe"
    else: return "Unlabeled"

COLOR_MAP_1 = {"Healthy": C_HEALTHY_TRAIN, "Severe": C_SEVERE_TRAIN, "Unlabeled": C_UNLAB_TRAIN}
ORDER_1 = ["Healthy", "Severe", "Unlabeled"]

def get_label_plot2(row):
    # Holdout always shown as predicted (dark)
    is_test = (row["Split"] == "Holdout Test")
    y_val = int(row["y"])

    if is_test:
        if y_val == 0: return "<b>Healthy Predicted</b>"
        elif y_val == 1: return "<b>Severe Predicted</b>"
        else: return "Unlabeled"

    # Train/Val shown as ground truth (light)
    if y_val == 0: return "Healthy"
    elif y_val == 1: return "Severe"
    else: return "Unlabeled"

COLOR_MAP_2 = {
    "Healthy": C_HEALTHY_TRAIN,
    "Severe": C_SEVERE_TRAIN,
    "Unlabeled": C_UNLAB_TRAIN,
    "<b>Healthy Predicted</b>": C_HEALTHY_PRED,
    "<b>Severe Predicted</b>": C_SEVERE_PRED
}

ORDER_2 = ["Healthy", "<b>Healthy Predicted</b>", "Severe", "<b>Severe Predicted</b>", "Unlabeled"]


axis_style = dict(
    title_font=dict(size=12, color="black"),
    showgrid=True, gridcolor="#ADD8E6", gridwidth=1,
    zeroline=True, showbackground=True, backgroundcolor=BG_COLOR,
    showticklabels=True, linecolor="#1e87a9", showline=True, mirror=True
)

layout_style = dict(
    margin=dict(l=0, r=0, b=0, t=0),
    scene=dict(
        bgcolor=BG_COLOR,
        xaxis=dict(title=dict(text="UMAP 1"), **axis_style),
        yaxis=dict(title=dict(text="UMAP 2"), **axis_style),
        zaxis=dict(title=dict(text="UMAP 3"), **axis_style)
    ),
    legend=dict(
        yanchor="middle", y=0.75, xanchor="right", x=0.70,
        bgcolor="rgba(255,255,255,0.8)", bordercolor="#ADD8E6",
        borderwidth=1, font=dict(size=12)
    )
)

# ============================================================
# 4) PLOT 1: UMAP in ORIGINAL INPUT SPACE
# ============================================================

X_fit = x_tensor[idx_fit].cpu().numpy()
X_test = x_tensor[idx_test_lbl].cpu().numpy()

umap_input = umap.UMAP(**umap_params_input)
U_fit_in = umap_input.fit_transform(X_fit)
U_test_in = umap_input.transform(X_test)

U_all_in = np.vstack([U_fit_in, U_test_in])
idx_all_in = np.concatenate([idx_fit, idx_test_lbl])

df_in = pd.DataFrame(U_all_in, columns=["UMAP1", "UMAP2", "UMAP3"])
df_in["Mouse"] = df_scaled_with_asym.loc[idx_all_in, "Mouse"].astype(str).values
df_in["Split"] = ["Train/Val"] * len(idx_fit) + ["Holdout Test"] * len(idx_test_lbl)
df_in["y"] = y_all_np[idx_all_in]
df_in["Group"] = df_in.apply(get_label_plot1, axis=1)

fig_in = px.scatter_3d(
    df_in, x="UMAP1", y="UMAP2", z="UMAP3",
    color="Group", color_discrete_map=COLOR_MAP_1,
    category_orders={"Group": ORDER_1},
    hover_data=["Mouse", "Split", "Group"],
    title=""
)
fig_in.update_traces(marker=dict(size=4.5, line=dict(width=0.2, color="black")))
fig_in.update_layout(**layout_style)

save_path_in = "/Users/asm/Desktop/UNI/MasterThesis/holdout_cvae1.html"
pio.write_html(fig_in, file=save_path_in, auto_open=False)
print("Saved input-space UMAP:", save_path_in)

# ============================================================
# 5) PLOT 2: LATENT SPACE -> supervised UMAP (same coords for plots 2/3/4)
# ============================================================

@torch.no_grad()
def encode_mu(indices):
    X = x_tensor[indices].to(device)
    A = asymmetry_tensor[indices].to(device)
    mu, _ = model.encode(X, y=None, asym=A)
    return mu.detach().cpu().numpy()

Z_fit = encode_mu(idx_fit)
Z_test = encode_mu(idx_test_lbl)

y_fit = y_all_np[idx_fit]  # supervised labels for fit set

print("Running supervised UMAP on latent vectors...")
umap_lat = umap.UMAP(**umap_params_supervised)
umap_lat.fit(Z_fit, y=y_fit)

U_fit_lat = umap_lat.transform(Z_fit)
U_test_lat = umap_lat.transform(Z_test)

U_all_lat = np.vstack([U_fit_lat, U_test_lat])
Z_total = np.vstack([Z_fit, Z_test])
idx_all_lat = np.concatenate([idx_fit, idx_test_lbl])

df_lat = pd.DataFrame(U_all_lat, columns=["UMAP1", "UMAP2", "UMAP3"])
df_lat["Mouse"] = df_scaled_with_asym.loc[idx_all_lat, "Mouse"].astype(str).values
df_lat["Split"] = ["Train/Val"] * len(idx_fit) + ["Holdout Test"] * len(idx_test_lbl)
df_lat["y"] = y_all_np[idx_all_lat]
df_lat["Group"] = df_lat.apply(get_label_plot2, axis=1)

# sanity check: Train/Val Healthy must be 31
print("Sanity check — Train/Val Healthy points:",
      ((df_lat["Split"]=="Train/Val") & (df_lat["Group"]=="Healthy")).sum())

fig_lat = px.scatter_3d(
    df_lat, x="UMAP1", y="UMAP2", z="UMAP3",
    color="Group", color_discrete_map=COLOR_MAP_2,
    category_orders={"Group": ORDER_2},
    hover_data=["Mouse", "Split", "Group"],
    title="Supervised UMAP (Latent Space)"
)
fig_lat.update_traces(marker=dict(size=4.5, line=dict(width=0.2, color="black")))
fig_lat.update_layout(**layout_style)

save_path_lat = "/Users/asm/Desktop/UNI/MasterThesis/holdout_cvae2.html"
pio.write_html(fig_lat, file=save_path_lat, auto_open=True)
print("Saved latent-space UMAP:", save_path_lat)

# ============================================================
# 6) METADATA EXTRACTION (for State plot) + DBSCAN on Z_total
# ============================================================

def extract_metadata(row):
    dataset_str = str(row.get("Dataset", "Unknown_Unknown"))
    parts = dataset_str.split("_")
    raw_treat = parts[0]
    raw_state = parts[1] if len(parts) > 1 else "Unknown"

    # Treatment
    if "Trained" in raw_treat: treatment = "Other"
    elif "Nimo" in raw_treat: treatment = "Nimo"
    elif "Placebo" in raw_treat: treatment = "Placebo"
    else: treatment = "Other"

    # State mapping
    if raw_state == "PreSCI": state = "Healthy"
    elif raw_state == "2weeksPostSCI": state = "Acute"
    elif raw_state in ["8weeksPostSCI", "10wPostSCI", "16wPostSCI"]: state = "Chronic"
    else: state = raw_state

    return pd.Series([treatment, state])

# Apply metadata for exactly the points in df_lat (same UMAP, same 31 healthy)
meta = df_scaled_with_asym.loc[idx_all_lat].apply(extract_metadata, axis=1)
df_lat["Treatment"] = meta[0].values
df_lat["State"] = meta[1].values

# ============================================================
# 7) PLOT 3: DBSCAN CLUSTERS (computed on Z_total, displayed on same UMAP coords)
# ============================================================

print("Running DBSCAN on latent vectors (Z_total)...")
dbscan = DBSCAN(eps=0.65, min_samples=9)
cluster_labels = dbscan.fit_predict(Z_total)

# Reassign noise via 1-NN
mask_noise = cluster_labels == -1
mask_labeled = ~mask_noise
if np.any(mask_noise) and np.any(mask_labeled):
    print(f"Reassigning {np.sum(mask_noise)} noise points...")
    knn = KNeighborsClassifier(n_neighbors=1)
    knn.fit(Z_total[mask_labeled], cluster_labels[mask_labeled])
    cluster_labels[mask_noise] = knn.predict(Z_total[mask_noise])

df_lat["Cluster"] = cluster_labels.astype(str)
df_lat_db = df_lat.sort_values("Cluster")

# Color map for clusters (0/1 fixed, others pastel)
unique_clusters = sorted(df_lat_db["Cluster"].unique(), key=lambda x: int(x) if x.isdigit() else 999)
extra_colors = px.colors.qualitative.Pastel + px.colors.qualitative.Set3
color_iter = iter(extra_colors)
cluster_color_map = {}

for lab in unique_clusters:
    if lab == "2": cluster_color_map[lab] = "#F08080"    # red-ish
    elif lab == "1": cluster_color_map[lab] = "#6495ED"  # blue-ish
    else: cluster_color_map[lab] = next(color_iter)

fig_db = px.scatter_3d(
    df_lat_db,
    x="UMAP1", y="UMAP2", z="UMAP3",
    color="Cluster",
    color_discrete_map=cluster_color_map,
    hover_data=["Mouse", "Cluster", "State", "Split"],
    title=""
)
fig_db.update_traces(marker=dict(size=4.5, line=dict(width=0.5, color="black")))
fig_db.update_layout(**layout_style)
fig_db.update_layout(legend_title_text="DBSCAN Clusters")

save_path_db = "/Users/asm/Desktop/UNI/MasterThesis/holdoutDBSCAN.html"
pio.write_html(fig_db, file=save_path_db, auto_open=True)
print("Saved DBSCAN plot:", save_path_db)

# ============================================================
# 8) PLOT 4: BIOLOGICAL STATES (Healthy / Acute / Chronic) on same UMAP coords
# ============================================================

color_map_state = {
    "Healthy": "blue",
    "Acute": "lightcoral",
    "Chronic": "lightgreen",
    "Unknown": "gray"
}

fig_state = px.scatter_3d(
    df_lat,
    x="UMAP1", y="UMAP2", z="UMAP3",
    color="State",
    color_discrete_map=color_map_state,
    hover_data=["Mouse", "State", "Treatment", "Split"],
    title=""
)
fig_state.update_traces(marker=dict(size=4.5, line=dict(width=0.2, color="black")))
fig_state.update_layout(**layout_style)
fig_state.update_layout(legend_title_text="State")

save_path_st = "/Users/asm/Desktop/UNI/MasterThesis/holdoutstates.html"
pio.write_html(fig_state, file=save_path_st, auto_open=True)
print("Saved States plot:", save_path_st)

In [ ]:
print("\n========================================")
print("NUMBER OF MICE PER BIOLOGICAL STATE (State Plot)")
print("========================================")

# Restrict to Train/Val (since those are true biological states)
df_state_eval = df_lat[df_lat["Split"] == "Train/Val"].copy()

# Keep only biological states
df_state_eval = df_state_eval[df_state_eval["State"].isin(["Healthy", "Acute", "Chronic"])]

# Count UNIQUE mice per state
state_counts = df_state_eval.groupby("State")["Mouse"].nunique()

# Print nicely
for state in ["Healthy", "Acute", "Chronic"]:
    n = state_counts.get(state, 0)
    print(f"{state}: {n} mice")

print("----------------------------------------")
print(f"Total unique mice plotted: {df_state_eval['Mouse'].nunique()}")
print("========================================\n")

In [ ]:
# ==============================================================================
# 2D PLOT: TRAJECTORIES (Nimo vs Placebo) — choose UMAP axes
# ==============================================================================

import plotly.graph_objects as go
import numpy as np
import plotly.io as pio

print("Generating 2D Trajectories Plot...")

# -----------------------------
# CHOOSE WHICH UMAP AXES TO PLOT
# -----------------------------
x_axis = "UMAP1"
y_axis = "UMAP2"   # <- change to "UMAP3" if you want

# ------------------------------------------------------------------
# 0) Use df_lat from the previous script (latent UMAP master dataframe)
#    Restrict to Train/Val so we plot biological states (Healthy/Acute/Chronic)
# ------------------------------------------------------------------
df_base = df_lat[df_lat["Split"] == "Train/Val"].copy()

wanted_states = ["Healthy", "Acute", "Chronic"]
df_base = df_base[df_base["State"].isin(wanted_states)].copy()

# ------------------------------------------------------------------
# 1) Identify mice with ALL 3 timepoints (Healthy, Acute, Chronic)
# ------------------------------------------------------------------
mice_counts = df_base.groupby("Mouse")["State"].nunique()
valid_mice_ids = mice_counts[mice_counts == 3].index
df_valid = df_base[df_base["Mouse"].isin(valid_mice_ids)].copy()

if df_valid.empty:
    raise ValueError(
        "No mice found with all 3 states (Healthy, Acute, Chronic) in Train/Val."
    )

# ------------------------------------------------------------------
# 2) Select Random Mice (2 Placebo, 2 Nimo) with fixed seed
# ------------------------------------------------------------------
valid_placebo = df_valid[df_valid["Treatment"] == "Placebo"]["Mouse"].unique()
valid_nimo    = df_valid[df_valid["Treatment"] == "Nimo"]["Mouse"].unique()

rng = np.random.default_rng(23)
n_placebo = min(len(valid_placebo), 2)
n_nimo    = min(len(valid_nimo), 2)

sel_placebo = rng.choice(valid_placebo, n_placebo, replace=False) if n_placebo > 0 else np.array([])
sel_nimo    = rng.choice(valid_nimo, n_nimo, replace=False) if n_nimo > 0 else np.array([])
selected_mice = np.concatenate([sel_placebo, sel_nimo])

if len(selected_mice) == 0:
    raise ValueError("No valid mice available for Placebo/Nimo selection.")

# ------------------------------------------------------------------
# 3) Build trajectory df and sort by state order
# ------------------------------------------------------------------
df_traj = df_valid[df_valid["Mouse"].isin(selected_mice)].copy()

order_map = {"Healthy": 0, "Acute": 1, "Chronic": 2}
df_traj["TimeOrder"] = df_traj["State"].map(order_map)
df_traj = df_traj.sort_values(by=["Mouse", "TimeOrder"])

# Keep ONE point per mouse per state (prevents zig-zag if duplicates exist)
df_traj = df_traj.drop_duplicates(subset=["Mouse", "State"], keep="first")

# ------------------------------------------------------------------
# 4) Define state colors (markers)
# ------------------------------------------------------------------
color_map_state = {
    "Healthy": "blue",
    "Acute": "lightcoral",
    "Chronic": "lightgreen"
}

# ------------------------------------------------------------------
# 5) Build 2D figure
# ------------------------------------------------------------------
fig_traj_2d = go.Figure()

# A) Plot LINES first
for mouse in selected_mice:
    mouse_data = df_traj[df_traj["Mouse"] == mouse].sort_values("TimeOrder")
    if len(mouse_data) < 2:
        continue

    treat = mouse_data["Treatment"].iloc[0]
    if treat == "Nimo":
        l_color = "rgb(60, 60, 60)"
    else:
        l_color = "rgb(160, 160, 160)"

    fig_traj_2d.add_trace(go.Scatter(
        x=mouse_data[x_axis],
        y=mouse_data[y_axis],
        mode="lines",
        line=dict(color=l_color, width=3),
        showlegend=False,
        hoverinfo="skip"
    ))

# B) Plot DOTS (markers) by state
for state, color in color_map_state.items():
    subset = df_traj[df_traj["State"] == state]
    if subset.empty:
        continue

    fig_traj_2d.add_trace(go.Scatter(
        x=subset[x_axis],
        y=subset[y_axis],
        mode="markers",
        marker=dict(size=10, color=color, line=dict(width=1, color="black")),
        name=state,
        hovertext=subset["Mouse"],
        hoverinfo="text"
    ))

# Dummy legend entries for trajectory lines
fig_traj_2d.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
                                line=dict(color="rgb(60, 60, 60)", width=3), name="Nimo"))
fig_traj_2d.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
                                line=dict(color="rgb(160, 160, 160)", width=3), name="Placebo"))

# ------------------------------------------------------------------
# 6) Styling
# ------------------------------------------------------------------
axis_style_2d = dict(
    showgrid=True,
    gridcolor="#ADD8E6",
    gridwidth=1,
    zeroline=True,
    showline=True,
    linecolor="#1e87a9",
    mirror=True,
    title_font=dict(size=14, color="black"),
    tickfont=dict(size=12)
)

fig_traj_2d.update_layout(
    title="",
    width=800,
    height=700,
    plot_bgcolor="#F0F8FF",
    paper_bgcolor="white",
    xaxis=dict(title=x_axis.replace("UMAP", "UMAP "), **axis_style_2d),
    yaxis=dict(title=y_axis.replace("UMAP", "UMAP "), **axis_style_2d),
    legend=dict(
        yanchor="top",
        y=0.50,
        xanchor="right",
        x=0.98,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#ADD8E6",
        borderwidth=1
    )
)

# ------------------------------------------------------------------
# 7) Save
# ------------------------------------------------------------------
save_path_2d = "/Users/asm/Desktop/UNI/MasterThesis/holdout2dP.html"
pio.write_html(fig_traj_2d, save_path_2d, auto_open=True)

print(f"2D Trajectory Plot saved to: {save_path_2d}")
print("=" * 30)
print(f"SELECTED PLACEBO MICE: {sel_placebo}")
print(f"SELECTED NIMO MICE:    {sel_nimo}")
print("=" * 30)

In [ ]:
from sklearn.metrics import (
    silhouette_score, adjusted_rand_score, davies_bouldin_score, silhouette_samples
)
import numpy as np

print("========================================")
print("QUANTITATIVE EVALUATION (DBSCAN on Latent Space)")
print("========================================")

# ==============================================================================
# 1) PREPARE DATA
# ==============================================================================
# Ground truth labels aligned to Z_total / df_lat ordering
# (0=Healthy, 1=Severe, -1=Unlabeled)  <-- assumes that's how y_all_np is encoded
Y_total = y_all_np[idx_all_lat]

# Internal metrics: exclude DBSCAN noise points (-1)
mask_clustered = cluster_labels != -1

# Optionally restrict ARI to Train/Val only (recommended, since those are truly labeled)
# If your holdout labels are available and you want them included, set this to False.
ARI_TRAINVAL_ONLY = True
if ARI_TRAINVAL_ONLY:
    split_arr = df_lat["Split"].values
    mask_split = (split_arr == "Train/Val")
else:
    mask_split = np.ones_like(Y_total, dtype=bool)

# ==============================================================================
# 2) SILHOUETTE SCORE
# ==============================================================================
if np.sum(mask_clustered) > 1 and len(np.unique(cluster_labels[mask_clustered])) > 1:
    sil_score = silhouette_score(Z_total[mask_clustered], cluster_labels[mask_clustered])
    print(f"Silhouette Score:        {sil_score:.4f}  (Higher is better)")
else:
    print("Silhouette Score:        N/A (Need >=2 clusters and enough clustered points)")

# ==============================================================================
# 3) DAVIES-BOULDIN INDEX
# ==============================================================================
if np.sum(mask_clustered) > 1 and len(np.unique(cluster_labels[mask_clustered])) > 1:
    db_score = davies_bouldin_score(Z_total[mask_clustered], cluster_labels[mask_clustered])
    print(f"Davies-Bouldin Index:    {db_score:.4f}  (Lower is better)")
else:
    print("Davies-Bouldin Index:    N/A (Need >=2 clusters and enough clustered points)")

# ==============================================================================
# 4) ADJUSTED RAND INDEX (ARI): clusters vs true labels
# ==============================================================================
mask_known_truth = (Y_total != -1)
mask_eval_ari = mask_known_truth & mask_clustered & mask_split

if np.sum(mask_eval_ari) > 0 and len(np.unique(cluster_labels[mask_eval_ari])) > 1:
    ari_score = adjusted_rand_score(Y_total[mask_eval_ari], cluster_labels[mask_eval_ari])
    print(f"Adjusted Rand Index:     {ari_score:.4f}  (Higher is better)")
    print(f"  (Evaluated on {np.sum(mask_eval_ari)} labeled & clustered samples)")
else:
    print("Adjusted Rand Index:     N/A (No overlap, or only 1 cluster)")

print("========================================")

# ==============================================================================
# 5) INDIVIDUAL CLUSTER SILHOUETTE SCORES
# ==============================================================================
print("\n========================================")
print("INDIVIDUAL CLUSTER SILHOUETTE SCORES")
print("========================================")

if np.sum(mask_clustered) > 1 and len(np.unique(cluster_labels[mask_clustered])) > 1:
    X_clustered = Z_total[mask_clustered]
    labels_clustered = cluster_labels[mask_clustered]

    sample_silhouette_values = silhouette_samples(X_clustered, labels_clustered)

    # ---- Optional: your manual naming (update if your cluster IDs differ) ----
    cluster_names = {
        2: "Severe",
        1: "Healthy (Subgroup A)",
        0: "Healthy (Subgroup B)",
    }

    for i in sorted(np.unique(labels_clustered)):
        vals = sample_silhouette_values[labels_clustered == i]
        print(f"Cluster {i} ({cluster_names.get(int(i), 'Unnamed')}): {np.mean(vals):.4f}")
        print(f"   -> Size: {len(vals)} points")

    # Optional combined “Healthy” average if you define which clusters are healthy
    healthy_clusters = [0, 1]
    healthy_mask = np.isin(labels_clustered, healthy_clusters)
    if np.sum(healthy_mask) > 0:
        print("----------------------------------------")
        print(f"Combined Healthy Clusters ({healthy_clusters}) Avg Score: {np.mean(sample_silhouette_values[healthy_mask]):.4f}")

else:
    print("N/A (Need >=2 clusters and enough clustered points)")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.metrics import pairwise_distances
import numpy as np

print("========================================")
print("STATISTICAL OVERLAP ANALYSIS: ACUTE vs. CHRONIC (Latent Space)")
print("========================================")

# ---------------------------------------------------------
# 1) PREPARE DATA (aligned with Z_total and df_lat)
# ---------------------------------------------------------
# Recommended: evaluate only Train/Val since those are true biological labels
df_eval = df_lat[df_lat["Split"] == "Train/Val"].copy()

# Build masks in the ORIGINAL df_lat order (so they match Z_total rows)
mask_tv = (df_lat["Split"] == "Train/Val").values
mask_acute = (df_lat["State"] == "Acute").values & mask_tv
mask_chronic = (df_lat["State"] == "Chronic").values & mask_tv

Z_acute = Z_total[mask_acute]
Z_chronic = Z_total[mask_chronic]

if len(Z_acute) < 2 or len(Z_chronic) < 2:
    raise ValueError(
        f"Not enough samples: Acute={len(Z_acute)}, Chronic={len(Z_chronic)} "
        "(need at least 2 in each to run CV reliably)."
    )

X_sub = np.vstack([Z_acute, Z_chronic])
y_sub = np.concatenate([np.zeros(len(Z_acute)), np.ones(len(Z_chronic))])  # 0=Acute, 1=Chronic

print(f"Comparing {len(Z_acute)} Acute samples vs {len(Z_chronic)} Chronic samples (Train/Val only).")

# Use stratified CV; adapt folds to smallest class size
min_class = int(min(len(Z_acute), len(Z_chronic)))
n_splits = min(5, min_class)
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# ==============================================================================
# TEST 1: LOGISTIC REGRESSION ACCURACY (Separability)
# ==============================================================================
clf = LogisticRegression(solver="liblinear", random_state=42)
scores = cross_val_score(clf, X_sub, y_sub, cv=cv)
mean_acc = scores.mean()

print(f"\n1. CLASSIFIER ACCURACY (LogReg, {n_splits}-fold CV)")
print(f"   Accuracy: {mean_acc:.4f} (Random chance is 0.50)")

if mean_acc < 0.6:
    print("   -> RESULT: High Overlap (Indistinguishable)")
elif mean_acc < 0.8:
    print("   -> RESULT: Moderate Overlap (Transitioning states)")
else:
    print("   -> RESULT: Distinct Clusters (Strong separation)")

# ==============================================================================
# TEST 2: PERMUTATION TEST (Centroid distance significance)
# ==============================================================================
def get_centroid_dist(data, labels):
    c0 = data[labels == 0].mean(axis=0)
    c1 = data[labels == 1].mean(axis=0)
    return np.linalg.norm(c0 - c1)

obs_dist = get_centroid_dist(X_sub, y_sub)

n_permutations = 1000
count_better = 0
rng = np.random.default_rng(42)

for _ in range(n_permutations):
    shuffled_y = rng.permutation(y_sub)
    rand_dist = get_centroid_dist(X_sub, shuffled_y)
    if rand_dist >= obs_dist:
        count_better += 1

p_value = (count_better + 1) / (n_permutations + 1)

print(f"\n2. PERMUTATION TEST (Centroid distance)")
print(f"   Observed Distance: {obs_dist:.4f}")
print(f"   P-Value:           {p_value:.4f}")

if p_value < 0.05:
    print("   -> RESULT: Significant Difference (p < 0.05)")
else:
    print("   -> RESULT: NO Significant Difference (p >= 0.05)")

print("========================================")

# ==============================================================================
# NON-LINEAR ANALYSIS
# ==============================================================================
print("\n========================================")
print("NON-LINEAR ANALYSIS: ACUTE vs. CHRONIC")
print("========================================")

# TEST 1: RANDOM FOREST
clf_rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
scores_rf = cross_val_score(clf_rf, X_sub, y_sub, cv=cv)
mean_acc_rf = scores_rf.mean()

print(f"\n1. RANDOM FOREST ACCURACY ({n_splits}-fold CV)")
print(f"   Accuracy: {mean_acc_rf:.4f} (Random chance is 0.50)")

if mean_acc_rf < 0.6:
    print("   -> RESULT: No Non-Linear Separation (Truly Identical)")
elif mean_acc_rf < 0.8:
    print("   -> RESULT: Weak/Moderate Separation")
else:
    print("   -> RESULT: Strong Non-Linear Separation")

# TEST 2: MMD (kernel two-sample test) + permutation p-value
def calculate_mmd(x, y, sigma=1.0):
    xx = rbf_kernel(x, x, gamma=1.0/(2*sigma**2))
    yy = rbf_kernel(y, y, gamma=1.0/(2*sigma**2))
    xy = rbf_kernel(x, y, gamma=1.0/(2*sigma**2))
    return xx.mean() + yy.mean() - 2 * xy.mean()

all_dists = pairwise_distances(X_sub, X_sub).flatten()
sigma_val = np.median(all_dists[all_dists > 0])  # heuristic

obs_mmd = calculate_mmd(Z_acute, Z_chronic, sigma=sigma_val)

n_permutations_mmd = 500
count_better = 0
rng = np.random.default_rng(42)

for _ in range(n_permutations_mmd):
    idx_perm = rng.permutation(len(X_sub))
    X_perm = X_sub[idx_perm]
    split_idx = len(Z_acute)
    g1 = X_perm[:split_idx]
    g2 = X_perm[split_idx:]
    rand_mmd = calculate_mmd(g1, g2, sigma=sigma_val)
    if rand_mmd >= obs_mmd:
        count_better += 1

p_value_mmd = (count_better + 1) / (n_permutations_mmd + 1)

print(f"\n2. MMD PERMUTATION TEST")
print(f"   Sigma (median heuristic): {sigma_val:.6f}")
print(f"   Observed MMD:             {obs_mmd:.6f}")
print(f"   P-Value:                  {p_value_mmd:.4f}")

if p_value_mmd < 0.05:
    print("   -> RESULT: Significant Distributional Difference (Shapes differ)")
else:
    print("   -> RESULT: Distributions are Statistically Indistinguishable")

print("========================================")

In [ ]:
import numpy as np
from scipy.stats import shapiro, levene, f_oneway, mannwhitneyu

print("\n========================================")
print("CENTROID DISPERSION ANALYSIS: ACUTE vs. CHRONIC")
print("========================================")

# 1. Compute Centroids (Mean position in latent space)
centroid_acute = Z_acute.mean(axis=0)
centroid_chronic = Z_chronic.mean(axis=0)

# 2. Calculate Euclidean distances from each point to its own group's centroid
dist_acute = np.linalg.norm(Z_acute - centroid_acute, axis=1)
dist_chronic = np.linalg.norm(Z_chronic - centroid_chronic, axis=1)

print(f"Mean distance to centroid (Acute):   {dist_acute.mean():.4f} ± {dist_acute.std():.4f}")
print(f"Mean distance to centroid (Chronic): {dist_chronic.mean():.4f} ± {dist_chronic.std():.4f}")
print("-" * 40)

# 3. Check Assumptions for ANOVA
alpha = 0.05

# Assumption A: Normality (Shapiro-Wilk Test)
# Null hypothesis: The data is normally distributed.
stat_sh_a, p_sh_a = shapiro(dist_acute)
stat_sh_c, p_sh_c = shapiro(dist_chronic)

is_normal_acute = p_sh_a > alpha
is_normal_chronic = p_sh_c > alpha
is_normal = is_normal_acute and is_normal_chronic

print("Assumption 1: Normality (Shapiro-Wilk)")
print(f"  Acute:   p = {p_sh_a:.4f} -> {'Pass' if is_normal_acute else 'Fail'}")
print(f"  Chronic: p = {p_sh_c:.4f} -> {'Pass' if is_normal_chronic else 'Fail'}")

# Assumption B: Homogeneity of Variances (Levene's Test)
# Null hypothesis: The groups have equal variances.
stat_lev, p_lev = levene(dist_acute, dist_chronic)
is_homoscedastic = p_lev > alpha

print("Assumption 2: Equal Variances (Levene's Test)")
print(f"  Test:    p = {p_lev:.4f} -> {'Pass' if is_homoscedastic else 'Fail'}")
print("-" * 40)

# 4. Statistical Testing Decision Tree
if is_normal and is_homoscedastic:
    # Both assumptions met -> Parametric Test
    print("DECISION: Assumptions met. Running One-Way ANOVA...")
    stat, p_val = f_oneway(dist_acute, dist_chronic)
    test_used = "One-Way ANOVA"
else:
    # One or both assumptions failed -> Non-Parametric Test
    print("DECISION: Assumptions violated. Running Mann-Whitney U test...")
    stat, p_val = mannwhitneyu(dist_acute, dist_chronic, alternative='two-sided')
    test_used = "Mann-Whitney U Test"

# 5. Final Results
print("\n" + "=" * 40)
print(f"FINAL RESULT ({test_used})")
print("=" * 40)
print(f"Statistic: {stat:.4f}")
print(f"P-Value:   {p_val:.4g}")

if p_val < alpha:
    print("-> CONCLUSION: Significant difference in cluster dispersion.")
    if dist_acute.mean() > dist_chronic.mean():
        print("   The Acute group is significantly more scattered than the Chronic group.")
    else:
        print("   The Chronic group is significantly more scattered than the Acute group.")
else:
    print("-> CONCLUSION: NO significant difference in cluster dispersion.")
    print("   Both groups have a similar spread in the latent space.")
print("========================================\n")